# Neural Networks

In [ ]:
%pip install pandas matplotlib seaborn scikit-learn tensorflow

In [2]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

data = pd.read_csv('/workspaces/group-project-bas-team/data/cleaned_data.csv')

train = data[data['order_year']<=2021]
test = data[data['order_year']>2021]

data_limited_train = train.drop(['Title', 'ASIN/ISBN (Product Code)'], axis=1)
data_limited_test = test.drop(['Title', 'ASIN/ISBN (Product Code)'], axis=1)

X_train = data_limited_train.drop('Category', axis=1)
y_train = data_limited_train['Category']
X_train_scaled = StandardScaler().fit_transform(X_train)

X_test = data_limited_test.drop('Category', axis=1)
y_test = data_limited_test['Category']
X_test_scaled = StandardScaler().fit_transform(X_test)

In [5]:
X_train_scaled.shape[1]

153

In [20]:
y_train.max()

np.int64(1624)

In [25]:
import tensorflow as tf

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(153, )))

model.add(tf.keras.layers.Dense(300, activation="relu")) 

model.add(tf.keras.layers.Dense(100, activation="relu"))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 

In [26]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 300)            │        46,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1625)           │       164,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 240,425 (939.16 KB)

 Trainable params: 240,425 (939.16 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

In [28]:
history = model.fit(X_train_scaled, y_train, epochs=30)

Epoch 1/30


W0000 00:00:1775400307.647810    8532 cpu_allocator_impl.cc:82] Allocation of 69556860 exceeds 10% of free system memory.


3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0588 - loss: 6.4077
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.0688 - loss: 6.0523
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.0755 - loss: 5.9403
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.0823 - loss: 5.8441
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.0868 - loss: 5.7572
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.0900 - loss: 5.6769
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.0934 - loss: 5.6044
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.0966 - loss: 5.5392
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.0995 - loss: 5.4807
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.1020 - loss: 5.4296
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - accuracy: 0.1041 - loss: 5.3830
Epoch 12/30
3552/3552 ━━━━━━━━

In [30]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1356/1356 - 4s - 3ms/step - accuracy: 0.0681 - loss: 5.9753

Test accuracy: 0.06811002641916275


Our initial neural network has an accuracy of about 6.8%, which is an improvement from our random forest model.

Next, we will attempt to build a wide & deep neural network

In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

normalization_layer = tf.keras.layers.Normalization()

# two Dense layers with 30 neurons each, using the ReLU activation function
hidden_layer1 = tf.keras.layers.Dense(30, activation="relu")
hidden_layer2 = tf.keras.layers.Dense(30, activation="relu")

concat_layer = tf.keras.layers.Concatenate()

output_layer = tf.keras.layers.Dense(1625)


input_ = tf.keras.layers.Input(shape=X_train_scaled.shape[1:])

normalized = normalization_layer(input_)

hidden1 = hidden_layer1(normalized)
hidden2 = hidden_layer2(hidden1)

concat = concat_layer([normalized, hidden2])

output = output_layer(concat)

model = tf.keras.Model(inputs=[input_], outputs=[output])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 153)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 153)       │        307 │ input_layer[0][0] │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 30)        │      4,620 │ normalization[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 30)        │        930 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 183)       │          0 │ normalization[0]… │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1625)      │    299,000 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 304,857 (1.16 MB)

 Trainable params: 304,550 (1.16 MB)

 Non-trainable params: 307 (1.20 KB)

In [35]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

normalization_layer.adapt(X_train_scaled)
history = model.fit(X_train_scaled, y_train, epochs=20)
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/20


W0000 00:00:1775401606.119216    8532 cpu_allocator_impl.cc:82] Allocation of 69556860 exceeds 10% of free system memory.


3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 8.6226e-04 - loss: 8.9416
Epoch 2/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 6.1590e-04 - loss: 8.2111
Epoch 3/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 5.2791e-04 - loss: 8.0843
Epoch 4/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 6.5109e-04 - loss: 8.0245
Epoch 5/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 21s 4ms/step - accuracy: 6.4229e-04 - loss: 8.0711
Epoch 6/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 2.2876e-04 - loss: 8.0465
Epoch 7/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 2.8155e-04 - loss: 7.9865
Epoch 8/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 2.3756e-04 - loss: 8.0215
Epoch 9/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 21s 4ms/step - accuracy: 2.9915e-04 - loss: 7.9450
Epoch 10/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 3.1675e-04 - loss: 7.9175
Epoch 11/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 2.9035e-04 -

Very poor accuracy and high loss - need to revise

In [53]:
X_train.columns

Index(['Purchase Price Per Unit', 'Quantity', 'age', 'hispanic', 'education',
       'income', 'howmany', 'hh-size', 'how-oft', 'order_month',
       ...
       'life-changes_Lost a job ,Divorce',
       'life-changes_Lost a job ,Divorce,Moved place of residence',
       'life-changes_Lost a job ,Had a child',
       'life-changes_Lost a job ,Moved place of residence',
       'life-changes_Lost a job ,Moved place of residence,Became pregnant',
       'life-changes_Lost a job ,Moved place of residence,Became pregnant,Had a child',
       'life-changes_Lost a job ,Moved place of residence,Had a child',
       'life-changes_Moved place of residence',
       'life-changes_Moved place of residence,Became pregnant,Had a child',
       'life-changes_Moved place of residence,Had a child'],
      dtype='str', length=153)

In [65]:
corr_matrix = X_train.corr().abs()
top_corr = corr_matrix.unstack().sort_values(ascending=False)
top_corr = top_corr[top_corr < 1].drop_duplicates()
print(top_corr.head(20))

diabetes_No                                 diabetes_Yes                 0.999812
wheelchair_Yes                              wheelchair_No                0.998180
state_Alabama                               Shipping Address State_AL    0.989636
Shipping Address State_ME                   state_Maine                  0.987357
state_Iowa                                  Shipping Address State_IA    0.982405
sexual-orientation_heterosexual (straight)  sexual-orientation_LGBTQ+    0.978118
Shipping Address State_NE                   state_Nebraska               0.973867
Shipping Address State_MI                   state_Michigan               0.972118
state_Idaho                                 Shipping Address State_ID    0.971907
marijuana_Yes                               marijuana_No                 0.969490
Shipping Address State_MO                   state_Missouri               0.969445
Shipping Address State_KY                   state_Kentucky               0.966610
state_North Caro

In [66]:
# It appears that states and shipping address states are highly correlated - we will remove all shipping address state variables
# Same with the diabetes_No, wheelchair_No, Marijuana_No, sexual-orientation_heterosexual (straight), gender_female, and alcohol_No variables, which are all very highly correlated with other variables
# this will significantly limit the number of variables to subset, making it easier to build our wide and deep neural network

X_train = X_train.drop(X_train.filter(regex='^Shipping Address').columns, axis=1)
X_test = X_test.drop(X_test.filter(regex='^Shipping Address').columns, axis=1)

In [69]:
X_train = X_train.drop(columns = ['diabetes_No', 'wheelchair_No', 'marijuana_No', 'sexual-orientation_heterosexual (straight)', 
                                  'gender_Female', 'alcohol_No'])
X_test = X_test.drop(columns = ['diabetes_No', 'wheelchair_No', 'marijuana_No', 'sexual-orientation_heterosexual (straight)', 
                                  'gender_Female', 'alcohol_No'])

In [70]:
X_train.columns

Index(['Purchase Price Per Unit', 'Quantity', 'age', 'hispanic', 'education',
       'income', 'howmany', 'hh-size', 'how-oft', 'order_month', 'order_day',
       'order_year', 'cigarettes_I stopped in the recent past',
       'cigarettes_No', 'cigarettes_Yes',
       'alcohol_I stopped in the recent past', 'alcohol_Yes',
       'marijuana_I stopped in the recent past', 'marijuana_Yes',
       'diabetes_Yes', 'wheelchair_Yes', 'gender_Male', 'gender_Other',
       'sexual-orientation_LGBTQ+', 'state_Alabama', 'state_Arizona',
       'state_Arkansas', 'state_California', 'state_Colorado',
       'state_Connecticut', 'state_Delaware', 'state_Florida', 'state_Georgia',
       'state_Hawaii', 'state_Idaho', 'state_Illinois', 'state_Indiana',
       'state_Iowa', 'state_Kansas', 'state_Kentucky', 'state_Louisiana',
       'state_Maine', 'state_Maryland', 'state_Massachusetts',
       'state_Michigan', 'state_Minnesota', 'state_Missouri', 'state_Nebraska',
       'state_Nevada', 'state_New H

In [71]:
X_test.columns

Index(['Purchase Price Per Unit', 'Quantity', 'age', 'hispanic', 'education',
       'income', 'howmany', 'hh-size', 'how-oft', 'order_month', 'order_day',
       'order_year', 'cigarettes_I stopped in the recent past',
       'cigarettes_No', 'cigarettes_Yes',
       'alcohol_I stopped in the recent past', 'alcohol_Yes',
       'marijuana_I stopped in the recent past', 'marijuana_Yes',
       'diabetes_Yes', 'wheelchair_Yes', 'gender_Male', 'gender_Other',
       'sexual-orientation_LGBTQ+', 'state_Alabama', 'state_Arizona',
       'state_Arkansas', 'state_California', 'state_Colorado',
       'state_Connecticut', 'state_Delaware', 'state_Florida', 'state_Georgia',
       'state_Hawaii', 'state_Idaho', 'state_Illinois', 'state_Indiana',
       'state_Iowa', 'state_Kansas', 'state_Kentucky', 'state_Louisiana',
       'state_Maine', 'state_Maryland', 'state_Massachusetts',
       'state_Michigan', 'state_Minnesota', 'state_Missouri', 'state_Nebraska',
       'state_Nevada', 'state_New H

In [73]:
X_train

,Purchase Price Per Unit,Quantity,age,hispanic,education,income,howmany,hh-size,how-oft,order_month,...,"life-changes_Lost a job ,Divorce","life-changes_Lost a job ,Divorce,Moved place of residence","life-changes_Lost a job ,Had a child","life-changes_Lost a job ,Moved place of residence","life-changes_Lost a job ,Moved place of residence,Became pregnant","life-changes_Lost a job ,Moved place of residence,Became pregnant,Had a child","life-changes_Lost a job ,Moved place of residence,Had a child",life-changes_Moved place of residence,"life-changes_Moved place of residence,Became pregnant,Had a child","life-changes_Moved place of residence,Had a child"
0,7.98,1,3,1,3,2,1,1,1,12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,13.99,1,3,1,3,2,1,1,1,12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,10.45,1,3,1,3,2,1,1,1,12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,10.00,1,3,1,3,2,1,1,1,12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,10.99,1,3,1,3,2,1,1,1,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156886,1089.99,1,2,0,2,2,1,2,1,12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
156887,15.95,1,2,0,2,2,1,2,1,12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
156888,8.99,1,2,0,2,2,1,2,1,12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
156889,52.49,1,2,0,2,2,1,2,1,12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [74]:
X_train_scaled = StandardScaler().fit_transform(X_train)
X_test_scaled = StandardScaler().fit_transform(X_test)

In [79]:
# Rerunning our neural networks to see if removing highly correlated variables improved our accuracy

tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(98, )))

model.add(tf.keras.layers.Dense(300, activation="relu")) 

model.add(tf.keras.layers.Dense(100, activation="relu"))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 

In [80]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 300)            │        29,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1625)           │       164,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 223,925 (874.71 KB)

 Trainable params: 223,925 (874.71 KB)

 Non-trainable params: 0 (0.00 B)

In [81]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

In [82]:
history = model.fit(X_train_scaled, y_train, epochs=30)

Epoch 1/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0574 - loss: 6.4430
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.0678 - loss: 6.0776
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0746 - loss: 5.9720
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0798 - loss: 5.8771
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0851 - loss: 5.7882
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0895 - loss: 5.7074
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0926 - loss: 5.6353
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - accuracy: 0.0954 - loss: 5.5715
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 21s 4ms/step - accuracy: 0.0974 - loss: 5.5149
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0995 - loss: 5.4647
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.1018 - loss: 5.4199
Epoch 12/30
3552/35

In [83]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1356/1356 - 5s - 4ms/step - accuracy: 0.0690 - loss: 5.9509

Test accuracy: 0.06900924444198608


In [84]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

normalization_layer = tf.keras.layers.Normalization()

# two Dense layers with 30 neurons each, using the ReLU activation function
hidden_layer1 = tf.keras.layers.Dense(30, activation="relu")
hidden_layer2 = tf.keras.layers.Dense(30, activation="relu")

concat_layer = tf.keras.layers.Concatenate()

output_layer = tf.keras.layers.Dense(1625)


input_ = tf.keras.layers.Input(shape=X_train_scaled.shape[1:])

normalized = normalization_layer(input_)

hidden1 = hidden_layer1(normalized)
hidden2 = hidden_layer2(hidden1)

concat = concat_layer([normalized, hidden2])

output = output_layer(concat)

# Finally create the model, specifying inputs and outputs
model = tf.keras.Model(inputs=[input_], outputs=[output])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 98)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 98)        │        197 │ input_layer[0][0] │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 30)        │      2,970 │ normalization[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 30)        │        930 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128)       │          0 │ normalization[0]… │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1625)      │    209,625 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 213,722 (834.86 KB)

 Trainable params: 213,525 (834.08 KB)

 Non-trainable params: 197 (792.00 B)

In [85]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

normalization_layer.adapt(X_train_scaled)
history = model.fit(X_train_scaled, y_train, epochs=20)
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0041 - loss: 9.4735
Epoch 2/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0030 - loss: 8.5759
Epoch 3/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 6.7749e-04 - loss: 8.4677
Epoch 4/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 6.8629e-04 - loss: 8.3299
Epoch 5/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 6.0710e-04 - loss: 8.2517
Epoch 6/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 5.7191e-04 - loss: 8.3056
Epoch 7/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 4.3993e-04 - loss: 8.3504
Epoch 8/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 4.3993e-04 - loss: 8.3052
Epoch 9/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 4.3993e-04 - loss: 8.2799
Epoch 10/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 4.3993e-04 - loss: 8.2830
Epoch 11/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 4.3993e-0

Next, we will try subsetting features so that the wide and deep components are trained on different features